<a href="https://colab.research.google.com/github/Svein-Tore/colab/blob/main/FOPDT-binder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import FloatSlider, Button, VBox, HBox, Output, FileUpload
from IPython.display import display, Markdown, HTML
import io
import base64

# === 1. Oppsett for filopplasting ===
uploader = FileUpload(accept='', multiple=False, description="Last opp fil")
main_output = Output()

def start_analysen(change):
    with main_output:
        main_output.clear_output()
        if not uploader.value:
            return

        # Henter fildata (robust metode for ipywidgets 7 og 8)
        if isinstance(uploader.value, dict):
            file_item = list(uploader.value.values())[0]
        else:
            file_item = uploader.value[0]

        content = file_item['content']
        df = pd.read_csv(io.BytesIO(content), sep=None, engine='python', decimal=',')

        tid_data = df.iloc[:,0].values
        niva_data = df.iloc[:,1].values

        # === 2. Automatisk estimering ===
        y0_est = niva_data[0]
        A_est = niva_data[-1] - y0_est
        thresh10 = y0_est + 0.1 * A_est
        thresh85 = y0_est + 0.85 * A_est
        thresh63 = y0_est + 0.63 * A_est

        idx10 = np.where(niva_data > thresh10)[0][0]
        idx85 = np.where(niva_data > thresh85)[0][0]
        idx63 = np.where(niva_data > thresh63)[0][0]

        L_est10 = tid_data[idx10]
        L_est85 = tid_data[idx85]
        L_est = max(0, abs(L_est10 - 0.05 * (L_est85 - L_est10)))
        T_est = max(0.1, tid_data[idx63] - L_est)

        # === 3. Plott-funksjon ===
        def plot_fopdt(A, T, L, y0):
            y_model = np.where(tid_data < L, y0, y0 + A * (1 - np.exp(-(tid_data - L) / T)))
            fig, ax = plt.subplots(figsize=(10, 5))
            ax.plot(tid_data, niva_data, "b.", markersize=3, label="Måledata")
            ax.plot(tid_data, y_model, "r-", linewidth=2, label=f"Modell: Δy={A:.2f}, T={T:.2f}")
            ax.set_xlabel("Tid [s]"); ax.set_ylabel("Nivå / Respons")
            ax.grid(True, which='both', linestyle='--', alpha=0.5); ax.legend()
            return fig

        # === 4. Widgets og Layout ===
        A_slider = FloatSlider(value=A_est, min=A_est*0.5, max=A_est*1.5, step=0.01, description="Δy")
        T_slider = FloatSlider(value=T_est, min=T_est*0.1, max=T_est*3, step=0.1, description="T")
        L_slider = FloatSlider(value=L_est, min=0, max=max(10, L_est*5), step=0.1, description="L")
        y0_slider = FloatSlider(value=y0_est, min=y0_est-2, max=y0_est+2, step=0.01, description="y0")

        save_btn = Button(description="Generer nedlasting", button_style='success')
        plot_out = Output()
        download_out = Output()

        def update_plot(change):
            with plot_out:
                plot_out.clear_output(wait=True)
                fig = plot_fopdt(A_slider.value, T_slider.value, L_slider.value, y0_slider.value)
                plt.show()

        def download_png(b):
            with download_out:
                download_out.clear_output()
                fig = plot_fopdt(A_slider.value, T_slider.value, L_slider.value, y0_slider.value)
                buf = io.BytesIO()
                fig.savefig(buf, format='png', dpi=300)
                plt.close(fig)
                buf.seek(0)
                b64 = base64.b64encode(buf.read()).decode()
                payload = f"data:image/png;base64,{b64}"
                html = f'<a download="FOPDT_modell.png" href="{payload}"><button style="width:100%; padding:10px; background:#28a745; color:white; border:none; border-radius:5px; cursor:pointer;">KLIKK HER FOR Å LAGRE PNG</button></a>'
                display(HTML(html))

        for s in [A_slider, T_slider, L_slider, y0_slider]:
            s.observe(update_plot, "value")
        save_btn.on_click(download_png)

        # --- Tabell og Dashbord ---
        tabell_html = """
        <div style="font-family: sans-serif; line-height: 1.4; margin-left: 40px; min-width: 350px;">
            <hr><h3>Instruksjoner</h3>
            <p>Tilpass modellen til måledataene.</p>
            <table style="width:100%; border-collapse: collapse; margin-top:10px;">
                <tr style="background-color: #f2f2f2;">
                    <th style="border: 1px solid #ddd; padding: 8px;">Valg av λ</th>
                    <th style="border: 1px solid #ddd; padding: 8px;">Regulering</th>
                </tr>
                <tr><td style="border: 1px solid #ddd; padding: 8px;">λ = T/2</td><td style="border: 1px solid #ddd; padding: 8px;">Rolig</td></tr>
                <tr><td style="border: 1px solid #ddd; padding: 8px;">λ = T/4</td><td style="border: 1px solid #ddd; padding: 8px;">Standard</td></tr>
                <tr><td style="border: 1px solid #ddd; padding: 8px;">λ = T/6</td><td style="border: 1px solid #ddd; padding: 8px;">Rask</td></tr>
            </table>
        </div>
        """

        slidere_kolonne = VBox([A_slider, T_slider, L_slider, y0_slider, save_btn, download_out], layout={'width': '250px', 'min_width': '250px'})
        tekst_kolonne = widgets.HTML(value=tabell_html)
        dashbord = HBox([plot_out, slidere_kolonne, tekst_kolonne], layout={'align_items': 'flex-start'})

        display(Markdown(f"**Autoestimat:** Δy={A_est:.2f}, T={T_est:.2f}, L={L_est:.2f}, y0={y0_est:.2f}"))
        display(dashbord)
        update_plot(None)

# === 5. Start programmet ===
uploader.observe(start_analysen, names='value')
display(Markdown("# FOPDT Modell-tilpasning"))
display(Markdown("### 1. Last opp måledata (.csv)"), uploader, main_output)